# Build a SQL agent
https://docs.langchain.com/oss/python/langchain/sql-agent

In [43]:
from dotenv import load_dotenv
load_dotenv()

import os
import sqlite3
from pathlib import Path

## 1. Select an LLM

In [44]:
from langchain_groq import ChatGroq
from ff_ai_assistant.config import GROQ_MODEL, GROQ_TEMPERATURE

model = ChatGroq(
    model=GROQ_MODEL,
    temperature=GROQ_TEMPERATURE,
)
model

test_msg = model.invoke("What team does Christian McCaffrey play for?")
print(test_msg.content)

<think>
Okay, so the user is asking which team Christian McCaffrey plays for. Let me start by recalling who Christian McCaffrey is. He's a professional football player, right? I think he's a running back. I remember he was with the San Francisco 49ers for a while. Wait, but I'm not sure if he's still there. Let me think.

I recall that in the 2022 season, the 49ers had some changes. Maybe he was traded or signed with another team. Oh, right! I think he was traded to the Carolina Panthers. Let me confirm that. The trade happened in 2022, so he joined the Panthers. But wait, was that the final move? Or did he end up with another team? I should check if there were any other trades or if he stayed with the Panthers. Also, considering the current year is 2023, so the latest season would be 2023. 

Wait, another thought: sometimes players get traded mid-season. Did McCaffrey get traded again in 2023? I don't recall any recent trades involving him. So, as of the latest information, he should 

## 2. Configure the database

In [45]:
import requests

# Download the Chinook.db file to ff_ai_assistant/data/
data_dir = Path("../data").resolve()
data_dir.mkdir(parents=True, exist_ok=True)  # Ensure the directory exists

local_path = data_dir / "Chinook.db"
print(local_path)

if local_path.exists():
    print(f"{local_path} already exists, skipping download.")
else:
    url = "https://storage.googleapis.com/benchmarks-artifacts/chinook/Chinook.db"
    response = requests.get(url)
    if response.status_code == 200:
        local_path.write_bytes(response.content)
        print(f"File downloaded and saved as {local_path}")
    else:
        print(f"Failed to download the file. Status code: {response.status_code}")

/home/phoenician/Projects/ff_ai_assistant/data/Chinook.db
/home/phoenician/Projects/ff_ai_assistant/data/Chinook.db already exists, skipping download.


In [46]:
# Create SQLDatabase wrapper
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///../data/Chinook.db")

print(f"Dialect: {db.dialect}")
print(f"Available tables: {db.get_usable_table_names()}")
print(f'Sample output: {db.run("SELECT * FROM Artist LIMIT 5;")}')

Dialect: sqlite
Available tables: ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']
Sample output: [(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains')]


## 3. Add tools for database interactions

In [47]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=db, llm=model)

tools = toolkit.get_tools()

for tool in tools:
    print(f"{tool.name}: {tool.description}\n")

sql_db_query: Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.

sql_db_schema: Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3

sql_db_list_tables: Input is an empty string, output is a comma-separated list of tables in the database.

sql_db_query_checker: Use this tool to double check if your query is correct before executing it. Always use this tool before executing a query with sql_db_query!



## 4. Use create_agent

In [ ]:
# Initialize the agent with a descriptive system prompt to customize its behavior
system_prompt = """
You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run,
then look at the results of the query and return the answer. Unless the user
specifies a specific number of examples they wish to obtain, always limit your
query to at most {top_k} results.

You can order the results by a relevant column to return the most interesting
examples in the database. Never query for all the columns from a specific table,
only ask for the relevant columns given the question.

You MUST double check your query before executing it. If you get an error while
executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the
database.

To start you should ALWAYS look at the tables in the database to see what you
can query. Do NOT skip this step.

Then you should query the schema of the most relevant tables.
""".format(
    dialect=db.dialect,
    top_k=5,
)

In [49]:
# Now, create an agent with the model, tools, and prompt
from langchain.agents import create_agent

agent = create_agent(
    model,
    tools,
    system_prompt=system_prompt,
)

## 5. Run the agent

In [50]:
# Run the agent on a sample query and observe its behavior
question = "Which genre on average has the longest tracks?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Which genre on average has the longest tracks?


================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (56ccrmhen)
 Call ID: 56ccrmhen
  Args:
    tool_input:
================================= Tool Message =================================
Name: sql_db_list_tables

Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (avanhgsx7)
 Call ID: avanhgsx7
  Args:
    table_names: Genre, Track
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE "Genre" (
	"GenreId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("GenreId")
)

/*
3 rows from Genre table:
GenreId	Name
1	Rock
2	Jazz
3	Metal
*/


CREATE TABLE "Track" (
	"TrackId" INTEGER NOT NULL, 
	"Name" NVARCHAR(200) NOT NULL, 
	"AlbumId" INTEGER, 
	"MediaTypeId" INTEGER NOT NULL, 
	"GenreId" INTEGER, 


## 6. Implement human-in-the-loop review

In [51]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model,
    tools,
    system_prompt=system_prompt,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"sql_db_query": True},
            description_prefix="Tool execution pending approval",
        ),
    ],
    checkpointer=InMemorySaver(),
)

# On running the agent, it will now pause for review before executing the sql_db_query tool:
question = "Which genre on average has the longest tracks?"
config = {"configurable": {"thread_id": "1"}}

print("Running agent...")
for step_idx, step in enumerate(agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    config,
    stream_mode="values",
)):
    print(f"\n--- Step {step_idx} ---")
    print(f"Step keys: {list(step.keys())}")
    if "__interrupt__" in step:
        print("INTERRUPTED (agent has paused before tool execution):")
        interrupt = step["__interrupt__"][0]
        for request in interrupt.value["action_requests"]:
            print(f"Interrupt Description: {request['description']}")
    elif "messages" in step:
        print("Current messages in this step:")
        step["messages"][-1].pretty_print()
    else:
        print("Other step (not interrupt, not messages):")
        print(step)

print("\n--- Resuming agent execution after approval ---")
# We can resume execution, in this case accepting the query, using Command:
from langgraph.types import Command

for step_idx, step in enumerate(agent.stream(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config,
    stream_mode="values",
)):
    print(f"\n--- Step {step_idx} ---")
    print(f"Step keys: {list(step.keys())}")
    if "messages" in step:
        print("Current messages in this step:")
        step["messages"][-1].pretty_print()
    elif "__interrupt__" in step:
        print("INTERRUPTED (agent has paused again):")
        interrupt = step["__interrupt__"][0]
        for request in interrupt.value["action_requests"]:
            print(f"Interrupt Description: {request['description']}")
    else:
        print("Other step (not interrupt, not messages):")
        print(step)

Running agent...

--- Step 0 ---
Step keys: ['messages']
Current messages in this step:
================================ Human Message =================================

Which genre on average has the longest tracks?



--- Step 1 ---
Step keys: ['messages']
Current messages in this step:
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (5g4zb42v4)
 Call ID: 5g4zb42v4
  Args:
    tool_input:

--- Step 2 ---
Step keys: ['messages']
Current messages in this step:
================================= Tool Message =================================
Name: sql_db_list_tables

Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track

--- Step 3 ---
Step keys: ['messages']
Current messages in this step:
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (c77edcgy0)
 Call ID: c77edcgy0
  Args:
    table_names: Genre, Track

--- Step 4 ---
Step keys: ['messages']
Current messages in this step:
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE "Genre" (
	"GenreId" INTEGER NOT NULL,

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `qwen/qwen3-32b` in organization `org_01kjqngqqsewmtkq0h5g86wgj1` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 4402, Requested 1618. Please try again in 200ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}